In [2]:

import pandas as pd

df = pd.read_excel("Online Retail.xlsx")

df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:

df.shape


(541909, 8)

In [5]:

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:

df.isnull().sum()


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:

(df.isnull().sum() / len(df) * 100).round(2)


InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

In [8]:

df.duplicated().sum()


np.int64(5268)

In [9]:

df[df.duplicated(keep=False)].head(10)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [10]:

# 원본 데이터 보존
df_clean = df.copy()

# 중복 행 제거
df_clean = df_clean.drop_duplicates()

# 제거 후 데이터 개수 확인
print("원본 데이터:", len(df))
print("중복 제거 후:", len(df_clean))
print("제거된 행:", len(df) - len(df_clean))


원본 데이터: 541909
중복 제거 후: 536641
제거된 행: 5268


In [11]:

# 취소 주문 여부 확인
df_clean["InvoiceNo"] = df_clean["InvoiceNo"].astype(str)

cancel = df_clean[
    df_clean["InvoiceNo"].str.startswith("C")
]

print("취소 거래 행 수:", len(cancel))

cancel.head()


취소 거래 행 수: 9251


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [12]:

# 취소 거래를 제외한 데이터
sales = df_clean[
    ~df_clean["InvoiceNo"].str.startswith("C")
].copy()

print("취소 거래 제외 후:", len(sales))


취소 거래 제외 후: 527390


In [13]:

print("수량 0 이하:", (sales["Quantity"] <= 0).sum())
print("가격 0 이하:", (sales["UnitPrice"] <= 0).sum())


수량 0 이하: 1336
가격 0 이하: 2512


In [14]:

# 정상 판매 데이터만 선택
sales = sales[
    (sales["Quantity"] > 0) &
    (sales["UnitPrice"] > 0)
].copy()

# 전처리 후 데이터 개수 확인
print("정상 판매 데이터:", len(sales))

# 데이터 앞부분 확인
sales.head()


정상 판매 데이터: 524878


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [15]:

# 매출 변수 생성
sales["Sales"] = sales["Quantity"] * sales["UnitPrice"]

# 결과 확인
sales[["Quantity", "UnitPrice", "Sales"]].head(10)


,Quantity,UnitPrice,Sales
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
5,2,7.65,15.30
6,6,4.25,25.50
7,6,1.85,11.10
8,6,1.85,11.10
9,32,1.69,54.08


In [16]:

print("전체 매출:", round(sales["Sales"].sum(), 2))
print("정상 판매 데이터 수:", len(sales))


전체 매출: 10642110.8
정상 판매 데이터 수: 524878


In [17]:

# 고객번호가 있는 거래만 선택
rfm_data = sales.dropna(
    subset=["CustomerID"]
).copy()

# 데이터 개수 확인
print("매출 분석용 데이터:", len(sales))
print("RFM 분석용 데이터:", len(rfm_data))


매출 분석용 데이터: 524878
RFM 분석용 데이터: 392692


In [18]:

# 고객번호를 문자형으로 변경
rfm_data["CustomerID"] = (
    rfm_data["CustomerID"]
    .astype(int)
    .astype(str)
)

# 결과 확인
rfm_data[["CustomerID", "InvoiceNo", "Sales"]].head()


,CustomerID,InvoiceNo,Sales
0,17850,536365,15.30
1,17850,536365,20.34
2,17850,536365,22.00
3,17850,536365,20.34
4,17850,536365,20.34


In [19]:

print("전체 고객 수:", rfm_data["CustomerID"].nunique())

print("전체 주문 수:", rfm_data["InvoiceNo"].nunique())

print("고객 분석용 매출:", round(rfm_data["Sales"].sum(), 2))


전체 고객 수: 4338
전체 주문 수: 18532
고객 분석용 매출: 8887208.89


In [20]:

from pathlib import Path

# 결과 저장 폴더 만들기
Path("output").mkdir(exist_ok=True)

# 매출 분석용 데이터 저장
sales.to_csv(
    "output/sales_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

# 고객 분석용 데이터 저장
rfm_data.to_csv(
    "output/rfm_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("데이터 저장 완료!")


데이터 저장 완료!
